In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-v0.1"
# base_model_id = "HuggingFaceH4/zephyr-7b-beta"
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
    # llm_int8_has_fp16_weight=True,
)
model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
from peft import PeftModel, LoraConfig, get_peft_model
peft_model_id = "mistral-nordavind-8bit-768/checkpoint-1000"
config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "lm_head"],
    bias="none",
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
)
peft_model = PeftModel.from_pretrained(model, peft_model_id, config=config).to(0)
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear8bitLt(
                (base_layer)

In [5]:
# publish peft:
peft_model.push_to_hub("tollefj/nordavind-8bit-768-lora-adaptor")

/work/tollefj/venv/lib/python3.8/site-packages/peft/utils/save_and_load.py:131: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


adapter_model.safetensors:   0%|          | 0.00/600M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/tollefj/nordavind-8bit-768-lora-adaptor/commit/de1c04366aadeae2ee679ef492827679c00839fa', commit_message='Upload model', commit_description='', oid='de1c04366aadeae2ee679ef492827679c00839fa', pr_url=None, pr_revision=None, pr_num=None)